In [3]:
!pip install tensorboard

   ---------------------------------------- 0.0/5.5 MB ? eta -:--:--
   ----------------- ---------------------- 2.4/5.5 MB 14.7 MB/s eta 0:00:01
   -------------------------------------- - 5.2/5.5 MB 14.3 MB/s eta 0:00:01
   ---------------------------------------- 5.5/5.5 MB 12.7 MB/s  0:00:00
   ---------------------------------------- 0.0/5.2 MB ? eta -:--:--
   ------------------------------ --------- 3.9/5.2 MB 18.8 MB/s eta 0:00:01
   ---------------------------------------- 5.2/5.2 MB 16.7 MB/s  0:00:00

  Attempting uninstall: protobuf

    Found existing installation: protobuf 5.29.3

   -------- ------------------------------- 1/5 [protobuf]
    Uninstalling protobuf-5.29.3:
   -------- ------------------------------- 1/5 [protobuf]
      Successfully uninstalled protobuf-5.29.3
   -------- ------------------------------- 1/5 [protobuf]
   -------- ------------------------------- 1/5 [protobuf]
   -------- ------------------------------- 1/5 [protobuf]
   -------- ----------

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
streamlit 1.51.0 requires pandas<3,>=1.4.0, but you have pandas 3.0.5 which is incompatible.
streamlit 1.51.0 requires protobuf<7,>=3.20, but you have protobuf 7.35.1 which is incompatible.


In [1]:
import warnings
warnings.filterwarnings(
    "ignore",
    category=UserWarning,
    message=r".*pkg_resources is deprecated as an API.*"
)

import pygame
import sys
from dataclasses import dataclass
from typing import List

# Configuración básica
WIDTH, HEIGHT = 800, 600
FPS = 60

# Colores
BLACK = (0, 0, 0)
WHITE = (240, 240, 240)
GRAY = (60, 60, 60)
BLUE = (80, 160, 255)
RED = (240, 90, 90)
GREEN = (80, 220, 120)
YELLOW = (250, 220, 100)
PURPLE = (170, 120, 250)
ORANGE = (255, 150, 70)

BLOCK_COLORS = [RED, ORANGE, YELLOW, GREEN, BLUE, PURPLE]

@dataclass
class Paddle:
    rect: pygame.Rect
    speed: int = 9

    def move(self, direction: int):
        self.rect.x += direction * self.speed
        self.rect.x = max(0, min(WIDTH - self.rect.width, self.rect.x))

    def draw(self, surf):
        pygame.draw.rect(surf, WHITE, self.rect, border_radius=6)

@dataclass
class Ball:
    x: float
    y: float
    vx: float
    vy: float
    r: int = 8
    stuck: bool = True  # comienza pegada a la paleta

    def rect(self) -> pygame.Rect:
        return pygame.Rect(int(self.x - self.r), int(self.y - self.r), self.r * 2, self.r * 2)

    def update(self):
        if not self.stuck:
            self.x += self.vx
            self.y += self.vy

            # Rebote en paredes
            if self.x - self.r <= 0 or self.x + self.r >= WIDTH:
                self.vx *= -1
                self.x = max(self.r, min(WIDTH - self.r, self.x))
            if self.y - self.r <= 0:
                self.vy *= -1
                self.y = self.r

    def draw(self, surf):
        pygame.draw.circle(surf, WHITE, (int(self.x), int(self.y)), self.r)

@dataclass
class Block:
    rect: pygame.Rect
    color: tuple
    alive: bool = True

    def draw(self, surf):
        if self.alive:
            pygame.draw.rect(surf, self.color, self.rect, border_radius=4)
            # borde
            pygame.draw.rect(surf, GRAY, self.rect, width=2, border_radius=4)

def create_blocks(cols=10, rows=6, margin=6, top=70, height=24):
    blocks: List[Block] = []
    total_margin_x = margin * (cols + 1)
    block_w = (WIDTH - total_margin_x) // cols
    for r in range(rows):
        y = top + r * (height + margin)
        color = BLOCK_COLORS[r % len(BLOCK_COLORS)]
        for c in range(cols):
            x = margin + c * (block_w + margin)
            rect = pygame.Rect(x, y, block_w, height)
            blocks.append(Block(rect, color))
    return blocks

def reflect_ball(ball: Ball, rect: pygame.Rect) -> None:
    # Determinar si el impacto es más horizontal o vertical
    brect = ball.rect()
    overlap = brect.clip(rect)
    if overlap.width < overlap.height:
        ball.vx *= -1
        if brect.centerx < rect.centerx:
            ball.x = rect.left - ball.r
        else:
            ball.x = rect.right + ball.r
    elif overlap.height < overlap.width:
        ball.vy *= -1
        if brect.centery < rect.centery:
            ball.y = rect.top - ball.r
        else:
            ball.y = rect.bottom + ball.r
    else:
        # esquina: invierte ambos
        ball.vx *= -1
        ball.vy *= -1

def main():
    pygame.init()
    screen = pygame.display.set_mode((WIDTH, HEIGHT))
    pygame.display.set_caption("Rompebloques")
    clock = pygame.time.Clock()
    font = pygame.font.SysFont(None, 28)
    bigfont = pygame.font.SysFont(None, 64)

    # Crear paleta y pelota
    paddle_w, paddle_h = 110, 16
    paddle = Paddle(pygame.Rect((WIDTH - paddle_w) // 2, HEIGHT - 60, paddle_w, paddle_h))

    ball = Ball(paddle.rect.centerx, paddle.rect.top - 10, 0, -6)

    # Bloques
    blocks = create_blocks()

    score = 0
    lives = 3
    running = True
    game_over = False
    win = False

    while running:
        dt = clock.tick(FPS)
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                running = False
            elif event.type == pygame.KEYDOWN:
                if event.key == pygame.K_ESCAPE:
                    running = False
                if event.key == pygame.K_SPACE and ball.stuck:
                    # Lanza la pelota con una velocidad horizontal según la posición en la paleta
                    rel = (ball.x - paddle.rect.centerx) / (paddle.rect.width / 2)
                    ball.vx = rel * 6
                    ball.vy = -6
                    ball.stuck = False

        # Entrada
        keys = pygame.key.get_pressed()
        direction = 0
        if keys[pygame.K_LEFT] or keys[pygame.K_a]:
            direction -= 1
        if keys[pygame.K_RIGHT] or keys[pygame.K_d]:
            direction += 1
        paddle.move(direction)

        # Si la pelota está pegada, sigue a la paleta
        if ball.stuck:
            ball.x = paddle.rect.centerx
            ball.y = paddle.rect.top - ball.r - 2

        # Actualizar pelota
        ball.update()

        # Colisión con paleta
        if not ball.stuck and ball.rect().colliderect(paddle.rect):
            # Ajuste de ángulo según punto de impacto
            hit_pos = (ball.x - paddle.rect.left) / paddle.rect.width  # 0..1
            ball.vx = (hit_pos - 0.5) * 10  # más a los lados => más ángulo
            ball.vy = -abs(ball.vy)
            # reposicionar para evitar quedar dentro
            ball.y = paddle.rect.top - ball.r - 1

        # Colisión con bloques
        if not ball.stuck:
            for block in blocks:
                if block.alive and ball.rect().colliderect(block.rect):
                    block.alive = False
                    score += 10
                    reflect_ball(ball, block.rect)
                    break  # una colisión por frame es suficiente

        # Caída de la pelota
        if ball.y - ball.r > HEIGHT:
            lives -= 1
            ball.stuck = True
            ball.x = paddle.rect.centerx
            ball.y = paddle.rect.top - ball.r - 2
            ball.vx, ball.vy = 0, -6
            if lives <= 0:
                game_over = True

        # Verificar victoria
        if all(not b.alive for b in blocks):
            win = True
            game_over = True

        # Dibujo
        screen.fill(BLACK)

        # Fondo sutil
        for y in range(0, HEIGHT, 40):
            pygame.draw.line(screen, (20, 20, 20), (0, y), (WIDTH, y))

        for b in blocks:
            b.draw(screen)

        paddle.draw(screen)
        ball.draw(screen)

        # HUD
        hud = font.render(f"Puntos: {score}   Vidas: {lives}", True, WHITE)
        screen.blit(hud, (16, 16))

        if game_over:
            msg = "¡Ganaste!" if win else "Juego terminado"
            txt = bigfont.render(msg, True, WHITE)
            sub = font.render("Pulsa R para reiniciar o Esc para salir", True, WHITE)
            screen.blit(txt, txt.get_rect(center=(WIDTH // 2, HEIGHT // 2 - 20)))
            screen.blit(sub, sub.get_rect(center=(WIDTH // 2, HEIGHT // 2 + 30)))

            pygame.display.flip()

            # Esperar reinicio o salir
            waiting = True
            while waiting:
                for event in pygame.event.get():
                    if event.type == pygame.QUIT:
                        pygame.quit()
                        sys.exit()
                    elif event.type == pygame.KEYDOWN:
                        if event.key == pygame.K_ESCAPE:
                            pygame.quit()
                            sys.exit()
                        if event.key == pygame.K_r:
                            # Reiniciar
                            blocks = create_blocks()
                            score = 0
                            lives = 3
                            game_over = False
                            win = False
                            ball.stuck = True
                            ball.vx, ball.vy = 0, -6
                            waiting = False
                clock.tick(30)
            continue

        pygame.display.flip()

    pygame.quit()
    sys.exit()

if __name__ == "__main__":
    main()


pygame 2.6.1 (SDL 2.28.4, Python 3.13.9)
Hello from the pygame community. https://www.pygame.org/contribute.html


SystemExit: 

C:\Users\User\anaconda3\Lib\site-packages\IPython\core\interactiveshell.py:3707: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [1]:
import warnings
warnings.filterwarnings(
    "ignore",
    category=UserWarning,
    message=r".*pkg_resources is deprecated as an API.*"
)

import sys
import pygame
import numpy as np
from dataclasses import dataclass
from typing import List

import gymnasium as gym
from gymnasium import spaces
from stable_baselines3 import PPO

# ==========================================
# 1. CONSTANTES Y CLASES DEL JUEGO
# ==========================================
WIDTH, HEIGHT = 800, 600

BLACK = (0, 0, 0)
WHITE = (240, 240, 240)
GRAY = (60, 60, 60)
BLUE = (80, 160, 255)
RED = (240, 90, 90)
GREEN = (80, 220, 120)
YELLOW = (250, 220, 100)
PURPLE = (170, 120, 250)
ORANGE = (255, 150, 70)

BLOCK_COLORS = [RED, ORANGE, YELLOW, GREEN, BLUE, PURPLE]

@dataclass
class Paddle:
    rect: pygame.Rect
    speed: int = 9

    def move(self, direction: int):
        self.rect.x += direction * self.speed
        self.rect.x = max(0, min(WIDTH - self.rect.width, self.rect.x))

    def draw(self, surf):
        pygame.draw.rect(surf, WHITE, self.rect, border_radius=6)

@dataclass
class Ball:
    x: float
    y: float
    vx: float
    vy: float
    r: int = 8
    stuck: bool = True

    def rect(self) -> pygame.Rect:
        return pygame.Rect(int(self.x - self.r), int(self.y - self.r), self.r * 2, self.r * 2)

    def update(self):
        if not self.stuck:
            self.x += self.vx
            self.y += self.vy

            if self.x - self.r <= 0 or self.x + self.r >= WIDTH:
                self.vx *= -1
                self.x = max(self.r, min(WIDTH - self.r, self.x))
            if self.y - self.r <= 0:
                self.vy *= -1
                self.y = self.r

    def draw(self, surf):
        pygame.draw.circle(surf, WHITE, (int(self.x), int(self.y)), self.r)

@dataclass
class Block:
    rect: pygame.Rect
    color: tuple
    alive: bool = True

    def draw(self, surf):
        if self.alive:
            pygame.draw.rect(surf, self.color, self.rect, border_radius=4)
            pygame.draw.rect(surf, GRAY, self.rect, width=2, border_radius=4)

def create_blocks(cols=10, rows=6, margin=6, top=70, height=24):
    blocks: List[Block] = []
    total_margin_x = margin * (cols + 1)
    block_w = (WIDTH - total_margin_x) // cols
    for r in range(rows):
        y = top + r * (height + margin)
        color = BLOCK_COLORS[r % len(BLOCK_COLORS)]
        for c in range(cols):
            x = margin + c * (block_w + margin)
            rect = pygame.Rect(x, y, block_w, height)
            blocks.append(Block(rect, color))
    return blocks

def reflect_ball(ball: Ball, rect: pygame.Rect) -> None:
    brect = ball.rect()
    overlap = brect.clip(rect)
    if overlap.width < overlap.height:
        ball.vx *= -1
        if brect.centerx < rect.centerx:
            ball.x = rect.left - ball.r
        else:
            ball.x = rect.right + ball.r
    elif overlap.height < overlap.width:
        ball.vy *= -1
        if brect.centery < rect.centery:
            ball.y = rect.top - ball.r
        else:
            ball.y = rect.bottom + ball.r
    else:
        ball.vx *= -1
        ball.vy *= -1

# ==========================================
# 2. ENTORNO CUSTOM PARA GYMNASIUM
# ==========================================
class BreakoutEnv(gym.Env):
    def __init__(self, render_mode=None):
        super().__init__()
        
        # Modo de visualización: None (entrenamiento rápido) o "human" (ver en pantalla)
        self.render_mode = render_mode
        self.screen = None
        self.clock = None

        # Acciones: 0 = Quieto, 1 = Izquierda, 2 = Derecha, 3 = Lanzar pelota
        self.action_space = spaces.Discrete(4)

        # Vector de Observación (Normalizado entre 0.0 y 1.0):
        # [paddle_x, ball_x, ball_y, ball_vx, ball_vy, ball_stuck, bloques_destruidos_ratio]
        self.observation_space = spaces.Box(
            low=0.0, high=1.0, shape=(7,), dtype=np.float32
        )

        self._init_game()

    def _init_game(self):
        paddle_w, paddle_h = 110, 16
        self.paddle = Paddle(pygame.Rect((WIDTH - paddle_w) // 2, HEIGHT - 60, paddle_w, paddle_h))
        self.ball = Ball(self.paddle.rect.centerx, self.paddle.rect.top - 10, 0, -6)
        self.blocks = create_blocks()
        self.total_blocks = len(self.blocks)
        self.score = 0
        self.lives = 3

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self._init_game()
        # Lanzamiento automático opcional al inicio para simplificar el aprendizaje
        self.ball.stuck = False
        self.ball.vx = 4.0
        self.ball.vy = -6.0

        observation = self._get_obs()
        return observation, {}

    def _get_obs(self):
        paddle_x_norm = self.paddle.rect.centerx / WIDTH
        ball_x_norm = self.ball.x / WIDTH
        ball_y_norm = self.ball.y / HEIGHT
        ball_vx_norm = (self.ball.vx + 10) / 20.0
        ball_vy_norm = (self.ball.vy + 10) / 20.0
        stuck_norm = 1.0 if self.ball.stuck else 0.0
        
        destroyed_blocks = sum(1 for b in self.blocks if not b.alive)
        blocks_ratio = destroyed_blocks / self.total_blocks

        return np.array([
            paddle_x_norm,
            ball_x_norm,
            ball_y_norm,
            ball_vx_norm,
            ball_vy_norm,
            stuck_norm,
            blocks_ratio
        ], dtype=np.float32)

    def step(self, action):
        reward = 0.0
        terminated = False
        truncated = False

        # --- 1. PROCESAR ACCIÓN ---
        if action == 1:   # Izquierda
            self.paddle.move(-1)
        elif action == 2: # Derecha
            self.paddle.move(1)
        elif action == 3 and self.ball.stuck: # Lanzar
            rel = (self.ball.x - self.paddle.rect.centerx) / (self.paddle.rect.width / 2)
            self.ball.vx = rel * 6
            self.ball.vy = -6
            self.ball.stuck = False

        # Mover pelota pegada a la paleta
        if self.ball.stuck:
            self.ball.x = self.paddle.rect.centerx
            self.ball.y = self.paddle.rect.top - self.ball.r - 2

        # Actualizar posición física
        self.ball.update()

        # --- 2. COLISIONES Y RECOMPENSAS ---
        # Colisión con Paleta
        if not self.ball.stuck and self.ball.rect().colliderect(self.paddle.rect):
            hit_pos = (self.ball.x - self.paddle.rect.left) / self.paddle.rect.width
            self.ball.vx = (hit_pos - 0.5) * 10
            self.ball.vy = -abs(self.ball.vy)
            self.ball.y = self.paddle.rect.top - self.ball.r - 1
            reward += 0.2  # Recompensa por atajar la pelota

        # Colisión con Bloques
        if not self.ball.stuck:
            for block in self.blocks:
                if block.alive and self.ball.rect().colliderect(block.rect):
                    block.alive = False
                    self.score += 10
                    reward += 2.0  # Recompensa por romper un bloque
                    reflect_ball(self.ball, block.rect)
                    break

        # Caída de la Pelota
        if self.ball.y - self.ball.r > HEIGHT:
            self.lives -= 1
            reward -= 5.0  # Penalización por perder una vida
            
            if self.lives <= 0:
                terminated = True
                reward -= 10.0
            else:
                self.ball.stuck = False
                self.ball.x = self.paddle.rect.centerx
                self.ball.y = self.paddle.rect.top - self.ball.r - 2
                self.ball.vx, self.ball.vy = 4, -6

        # Condición de Victoria
        if all(not b.alive for b in self.blocks):
            reward += 100.0
            terminated = True

        # Recompensa por proximidad (ayuda a acelerar el aprendizaje inicial)
        dist_x = abs(self.paddle.rect.centerx - self.ball.x) / WIDTH
        reward += (1.0 - dist_x) * 0.01

        if self.render_mode == "human":
            self.render()

        return self._get_obs(), reward, terminated, truncated, {}

    def render(self):
        if self.screen is None:
            pygame.init()
            if self.render_mode == "human":
                self.screen = pygame.display.set_mode((WIDTH, HEIGHT))
                pygame.display.set_caption("Rompebloques - Entrenamiento IA")
            else:
                self.screen = pygame.Surface((WIDTH, HEIGHT))
        
        if self.clock is None:
            self.clock = pygame.time.Clock()

        self.screen.fill(BLACK)
        for y in range(0, HEIGHT, 40):
            pygame.draw.line(self.screen, (20, 20, 20), (0, y), (WIDTH, y))

        for b in self.blocks:
            b.draw(self.screen)

        self.paddle.draw(self.screen)
        self.ball.draw(self.screen)

        if self.render_mode == "human":
            pygame.event.pump()
            pygame.display.flip()
            self.clock.tick(60)

    def close(self):
        if self.screen is not None:
            pygame.quit()
            self.screen = None


# ==========================================
# 3. ENTRENAMIENTO DE LA IA
# ==========================================
if __name__ == "__main__":
    # Instanciar el entorno sin render para máxima velocidad
    env = BreakoutEnv(render_mode=None)

    # Crear el modelo PPO (Multi-Layer Perceptron)
    model = PPO("MlpPolicy", env, verbose=1, learning_rate=0.0003, tensorboard_log="./tb_logs/")

    print("Iniciando entrenamiento...")
    # Entrenar durante 200,000 pasos (puedes aumentarlo a 500k para mejor rendimiento)
    model.learn(total_timesteps=200000)
    
    # Guardar el modelo entrenado
    model.save("bloques_ia_ppo")
    print("Modelo guardado exitosamente como 'bloques_ia_ppo.zip'.")
    env.close()

pygame 2.6.1 (SDL 2.28.4, Python 3.13.9)
Hello from the pygame community. https://www.pygame.org/contribute.html
Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
Iniciando entrenamiento...
Logging to ./tb_logs/PPO_1
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 379      |
|    ep_rew_mean     | -14.5    |
| time/              |          |
|    fps             | 576      |
|    iterations      | 1        |
|    time_elapsed    | 3        |
|    total_timesteps | 2048     |
---------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 370         |
|    ep_rew_mean          | -14.3       |
| time/                   |             |
|    fps                  | 405         |
|    iterations           | 2           |
|    time_elapsed         | 10          |
|    total_timesteps      | 4096        |
| train/             

In [5]:
# Cargar el modelo entrenado
model = PPO.load("bloques_ia_ppo")

# Crear el entorno con la ventana de Pygame activa
env = BreakoutEnv(render_mode="human")
obs, _ = env.reset()

for _ in range(5000):
    # La IA selecciona la mejor acción
    action, _states = model.predict(obs, deterministic=True)
    obs, reward, terminated, truncated, info = env.step(action)
    
    if terminated or truncated:
        obs, _ = env.reset()

env.close()

# Cambio de castigo y premio: se aumentan el premio y se aumenta el catigo 


In [19]:
import warnings
warnings.filterwarnings(
    "ignore",
    category=UserWarning,
    message=r".*pkg_resources is deprecated as an API.*"
)

import sys
import pygame
import numpy as np
from dataclasses import dataclass
from typing import List

import gymnasium as gym
from gymnasium import spaces
from stable_baselines3 import PPO

# ==========================================
# 1. CONSTANTES Y CLASES DEL JUEGO
# ==========================================
WIDTH, HEIGHT = 800, 600

BLACK = (0, 0, 0)
WHITE = (240, 240, 240)
GRAY = (60, 60, 60)
BLUE = (80, 160, 255)
RED = (240, 90, 90)
GREEN = (80, 220, 120)
YELLOW = (250, 220, 100)
PURPLE = (170, 120, 250)
ORANGE = (255, 150, 70)

BLOCK_COLORS = [RED, ORANGE, YELLOW, GREEN, BLUE, PURPLE]

@dataclass
class Paddle:
    rect: pygame.Rect
    speed: int = 9

    def move(self, direction: int):
        self.rect.x += direction * self.speed
        self.rect.x = max(0, min(WIDTH - self.rect.width, self.rect.x))

    def draw(self, surf):
        pygame.draw.rect(surf, WHITE, self.rect, border_radius=6)

@dataclass
class Ball:
    x: float
    y: float
    vx: float
    vy: float
    r: int = 8
    stuck: bool = True

    def rect(self) -> pygame.Rect:
        return pygame.Rect(int(self.x - self.r), int(self.y - self.r), self.r * 2, self.r * 2)

    def update(self):
        if not self.stuck:
            self.x += self.vx
            self.y += self.vy

            if self.x - self.r <= 0 or self.x + self.r >= WIDTH:
                self.vx *= -1
                self.x = max(self.r, min(WIDTH - self.r, self.x))
            if self.y - self.r <= 0:
                self.vy *= -1
                self.y = self.r

    def draw(self, surf):
        pygame.draw.circle(surf, WHITE, (int(self.x), int(self.y)), self.r)

@dataclass
class Block:
    rect: pygame.Rect
    color: tuple
    alive: bool = True

    def draw(self, surf):
        if self.alive:
            pygame.draw.rect(surf, self.color, self.rect, border_radius=4)
            pygame.draw.rect(surf, GRAY, self.rect, width=2, border_radius=4)

def create_blocks(cols=10, rows=6, margin=6, top=70, height=24):
    blocks: List[Block] = []
    total_margin_x = margin * (cols + 1)
    block_w = (WIDTH - total_margin_x) // cols
    for r in range(rows):
        y = top + r * (height + margin)
        color = BLOCK_COLORS[r % len(BLOCK_COLORS)]
        for c in range(cols):
            x = margin + c * (block_w + margin)
            rect = pygame.Rect(x, y, block_w, height)
            blocks.append(Block(rect, color))
    return blocks

def reflect_ball(ball: Ball, rect: pygame.Rect) -> None:
    brect = ball.rect()
    overlap = brect.clip(rect)
    if overlap.width < overlap.height:
        ball.vx *= -1
        if brect.centerx < rect.centerx:
            ball.x = rect.left - ball.r
        else:
            ball.x = rect.right + ball.r
    elif overlap.height < overlap.width:
        ball.vy *= -1
        if brect.centery < rect.centery:
            ball.y = rect.top - ball.r
        else:
            ball.y = rect.bottom + ball.r
    else:
        ball.vx *= -1
        ball.vy *= -1

# ==========================================
# 2. ENTORNO CUSTOM PARA GYMNASIUM
# ==========================================
class BreakoutEnv(gym.Env):
    def __init__(self, render_mode=None):
        super().__init__()
        
        # Modo de visualización: None (entrenamiento rápido) o "human" (ver en pantalla)
        self.render_mode = render_mode
        self.screen = None
        self.clock = None

        # Acciones: 0 = Quieto, 1 = Izquierda, 2 = Derecha, 3 = Lanzar pelota
        self.action_space = spaces.Discrete(4)

        # Vector de Observación (Normalizado entre 0.0 y 1.0):
        # [paddle_x, ball_x, ball_y, ball_vx, ball_vy, ball_stuck, bloques_destruidos_ratio]
        self.observation_space = spaces.Box(
            low=0.0, high=1.0, shape=(7,), dtype=np.float32
        )

        self._init_game()

    def _init_game(self):
        paddle_w, paddle_h = 110, 16
        self.paddle = Paddle(pygame.Rect((WIDTH - paddle_w) // 2, HEIGHT - 60, paddle_w, paddle_h))
        self.ball = Ball(self.paddle.rect.centerx, self.paddle.rect.top - 10, 0, -6)
        self.blocks = create_blocks()
        self.total_blocks = len(self.blocks)
        self.score = 0
        self.lives = 3

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self._init_game()
        # Lanzamiento automático opcional al inicio para simplificar el aprendizaje
        self.ball.stuck = False
        self.ball.vx = 4.0
        self.ball.vy = -6.0

        observation = self._get_obs()
        return observation, {}

    def _get_obs(self):
        paddle_x_norm = self.paddle.rect.centerx / WIDTH
        ball_x_norm = self.ball.x / WIDTH
        ball_y_norm = self.ball.y / HEIGHT
        ball_vx_norm = (self.ball.vx + 10) / 20.0
        ball_vy_norm = (self.ball.vy + 10) / 20.0
        stuck_norm = 1.0 if self.ball.stuck else 0.0
        
        destroyed_blocks = sum(1 for b in self.blocks if not b.alive)
        blocks_ratio = destroyed_blocks / self.total_blocks

        return np.array([
            paddle_x_norm,
            ball_x_norm,
            ball_y_norm,
            ball_vx_norm,
            ball_vy_norm,
            stuck_norm,
            blocks_ratio
        ], dtype=np.float32)

    def step(self, action):
        reward = 0.0
        terminated = False
        truncated = False

        # --- 1. PROCESAR ACCIÓN --- (Se mantiene igual)
        if action == 1:   # Izquierda
            self.paddle.move(-1)
        elif action == 2: # Derecha
            self.paddle.move(1)
        # Eliminamos la acción 3 (lanzar) para simplificar: la pelota sale sola al resetear
        
        # Mover pelota pegada a la paleta (si aplica)
        if self.ball.stuck:
            self.ball.x = self.paddle.rect.centerx
            self.ball.y = self.paddle.rect.top - self.ball.r - 2

        # Actualizar posición física
        self.ball.update()

        # --- 2. COLISIONES Y NUEVAS RECOMPENSAS ---

        # A) Recompensa por SUPERVIVENCIA (pequeña recompensa por frame viva)
        # Esto le dice: "Estar viva es bueno"
        reward += 0.01 

        # B) Recompensa por proximidad (Reward Shaping) - CLAVE PARA QUE SE MUEVA
        # Calculamos la distancia horizontal entre el centro de la paleta y la pelota
        dist_x = abs(self.paddle.rect.centerx - self.ball.x)
        
        # Normalizamos la distancia (0.0 = encima, 1.0 = en el otro extremo)
        dist_norm = dist_x / WIDTH
        
        # Damos una recompensa inversamente proporcional a la distancia.
        # Si está justo debajo, recibe +0.05 por frame. Si está lejos, 0.
        reward += (1.0 - dist_norm) * 0.05

        # Colisión con Paleta (Aumentamos el premio)
        if not self.ball.stuck and self.ball.rect().colliderect(self.paddle.rect):
            hit_pos = (self.ball.x - self.paddle.rect.left) / self.paddle.rect.width
            self.ball.vx = (hit_pos - 0.5) * 10
            self.ball.vy = -abs(self.ball.vy)
            self.ball.y = self.paddle.rect.top - self.ball.r - 1
            reward += 10.0  # ANTES: 0.2 | AHORA: +10.0 por atajarla!

        # Colisión con Bloques (Se mantiene similar)
        if not self.ball.stuck:
            for block in self.blocks:
                if block.alive and self.ball.rect().colliderect(block.rect):
                    block.alive = False
                    self.score += 10
                    reward += 1.0  # Bajamos un poco esto para que atajar sea más importante
                    reflect_ball(self.ball, block.rect)
                    break

        # Caída de la Pelota (AUMENTAMOS MUCHO EL CASTIGO)
        if self.ball.y - self.ball.r > HEIGHT:
            self.lives -= 1
            
            # Penalización CRÍTICA por dejar caer la pelota
            reward -= 50.0  # ANTES: -5.0 | AHORA: -50.0! Perder la pelota duele mucho.
            
            if self.lives <= 0:
                terminated = True
                reward -= 100.0 # Castigo final por Game Over
            else:
                # Resetear pelota rápido para que siga aprendiendo
                self.ball.stuck = False
                self.ball.x = self.paddle.rect.centerx
                self.ball.y = self.paddle.rect.top - self.ball.r - 2
                self.ball.vx, self.ball.vy = 4, -6

        # Condición de Victoria (Se mantiene igual)
        if all(not b.alive for b in self.blocks):
            reward += 100.0
            terminated = True

        if self.render_mode == "human":
            self.render()

        return self._get_obs(), reward, terminated, truncated, {}
    def render(self):
        if self.screen is None:
            pygame.init()
            if self.render_mode == "human":
                self.screen = pygame.display.set_mode((WIDTH, HEIGHT))
                pygame.display.set_caption("Rompebloques - Entrenamiento IA")
            else:
                self.screen = pygame.Surface((WIDTH, HEIGHT))
        
        if self.clock is None:
            self.clock = pygame.time.Clock()

        self.screen.fill(BLACK)
        for y in range(0, HEIGHT, 40):
            pygame.draw.line(self.screen, (20, 20, 20), (0, y), (WIDTH, y))

        for b in self.blocks:
            b.draw(self.screen)

        self.paddle.draw(self.screen)
        self.ball.draw(self.screen)

        if self.render_mode == "human":
            pygame.event.pump()
            pygame.display.flip()
            self.clock.tick(60)

    def close(self):
        if self.screen is not None:
            pygame.quit()
            self.screen = None


# ==========================================
# 3. ENTRENAMIENTO DE LA IA
# ==========================================
if __name__ == "__main__":
    # Instanciar el entorno sin render para máxima velocidad
    env = BreakoutEnv(render_mode=None)

    # Crear el modelo PPO (Multi-Layer Perceptron)
    model = PPO("MlpPolicy", env, verbose=1, learning_rate=0.0003, tensorboard_log="./tb_logs/")

    print("Iniciando entrenamiento...")
    # Entrenar durante 200,000 pasos (puedes aumentarlo a 500k para mejor rendimiento)
    model.learn(total_timesteps=400000)
    
    # Guardar el modelo entrenado
    model.save("bloques_ia_ppo")
    print("Modelo guardado exitosamente como 'bloques_ia_ppo.zip'.")
    env.close()

Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
Iniciando entrenamiento...
Logging to ./tb_logs/PPO_7
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 478      |
|    ep_rew_mean     | -206     |
| time/              |          |
|    fps             | 1254     |
|    iterations      | 1        |
|    time_elapsed    | 1        |
|    total_timesteps | 2048     |
---------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 445         |
|    ep_rew_mean          | -212        |
| time/                   |             |
|    fps                  | 983         |
|    iterations           | 2           |
|    time_elapsed         | 4           |
|    total_timesteps      | 4096        |
| train/                  |             |
|    approx_kl            | 0.008075621 |
|    clip_fraction        | 0.0399      |
|    cli

# SE MUEVE MEJOR 

In [24]:
# Cargar el modelo entrenado
model = PPO.load("bloques_ia_ppo")

# Crear el entorno con la ventana de Pygame activa
env = BreakoutEnv(render_mode="human")
obs, _ = env.reset()

for _ in range(5000):
    # La IA selecciona la mejor acción
    action, _states = model.predict(obs, deterministic=True)
    obs, reward, terminated, truncated, info = env.step(action)
    
    if terminated or truncated:
        obs, _ = env.reset()

env.close()

#  256 \times 256 neuronas. Ampliar el Espacio de Observación (Observar el Tablero Completo)

In [14]:
import warnings
warnings.filterwarnings(
    "ignore",
    category=UserWarning,
    message=r".*pkg_resources is deprecated as an API.*"
)

import sys
import pygame
import numpy as np
from dataclasses import dataclass
from typing import List

import gymnasium as gym
from gymnasium import spaces
from stable_baselines3 import PPO

# ==========================================
# 1. CONSTANTES Y CLASES DEL JUEGO
# ==========================================
WIDTH, HEIGHT = 800, 600

BLACK = (0, 0, 0)
WHITE = (240, 240, 240)
GRAY = (60, 60, 60)
BLUE = (80, 160, 255)
RED = (240, 90, 90)
GREEN = (80, 220, 120)
YELLOW = (250, 220, 100)
PURPLE = (170, 120, 250)
ORANGE = (255, 150, 70)

BLOCK_COLORS = [RED, ORANGE, YELLOW, GREEN, BLUE, PURPLE]

@dataclass
class Paddle:
    rect: pygame.Rect
    speed: int = 9

    def move(self, direction: int):
        self.rect.x += direction * self.speed
        self.rect.x = max(0, min(WIDTH - self.rect.width, self.rect.x))

    def draw(self, surf):
        pygame.draw.rect(surf, WHITE, self.rect, border_radius=6)

@dataclass
class Ball:
    x: float
    y: float
    vx: float
    vy: float
    r: int = 8
    stuck: bool = True

    def rect(self) -> pygame.Rect:
        return pygame.Rect(int(self.x - self.r), int(self.y - self.r), self.r * 2, self.r * 2)

    def update(self):
        if not self.stuck:
            self.x += self.vx
            self.y += self.vy

            if self.x - self.r <= 0 or self.x + self.r >= WIDTH:
                self.vx *= -1
                self.x = max(self.r, min(WIDTH - self.r, self.x))
            if self.y - self.r <= 0:
                self.vy *= -1
                self.y = self.r

    def draw(self, surf):
        pygame.draw.circle(surf, WHITE, (int(self.x), int(self.y)), self.r)

@dataclass
class Block:
    rect: pygame.Rect
    color: tuple
    alive: bool = True

    def draw(self, surf):
        if self.alive:
            pygame.draw.rect(surf, self.color, self.rect, border_radius=4)
            pygame.draw.rect(surf, GRAY, self.rect, width=2, border_radius=4)

def create_blocks(cols=10, rows=6, margin=6, top=70, height=24):
    blocks: List[Block] = []
    total_margin_x = margin * (cols + 1)
    block_w = (WIDTH - total_margin_x) // cols
    for r in range(rows):
        y = top + r * (height + margin)
        color = BLOCK_COLORS[r % len(BLOCK_COLORS)]
        for c in range(cols):
            x = margin + c * (block_w + margin)
            rect = pygame.Rect(x, y, block_w, height)
            blocks.append(Block(rect, color))
    return blocks

def reflect_ball(ball: Ball, rect: pygame.Rect) -> None:
    brect = ball.rect()
    overlap = brect.clip(rect)
    if overlap.width < overlap.height:
        ball.vx *= -1
        if brect.centerx < rect.centerx:
            ball.x = rect.left - ball.r
        else:
            ball.x = rect.right + ball.r
    elif overlap.height < overlap.width:
        ball.vy *= -1
        if brect.centery < rect.centery:
            ball.y = rect.top - ball.r
        else:
            ball.y = rect.bottom + ball.r
    else:
        ball.vx *= -1
        ball.vy *= -1

# ==========================================
# 2. ENTORNO GYMNASIUM MEJORADO
# ==========================================
class BreakoutEnv(gym.Env):
    def __init__(self, render_mode=None):
        super().__init__()
        self.render_mode = render_mode
        self.screen = None
        self.clock = None

        # Acciones: 0 = Quieto, 1 = Izquierda, 2 = Derecha
        self.action_space = spaces.Discrete(3)

        # Observación: 6 variables de estado + 60 bloques individuales (1.0 = vivo, 0.0 = destruido)
        num_blocks = 60
        self.observation_space = spaces.Box(
            low=-1.0, high=1.0, shape=(6 + num_blocks,), dtype=np.float32
        )

        self._init_game()

    def _init_game(self):
        paddle_w, paddle_h = 110, 16
        self.paddle = Paddle(pygame.Rect((WIDTH - paddle_w) // 2, HEIGHT - 60, paddle_w, paddle_h))
        self.ball = Ball(self.paddle.rect.centerx, self.paddle.rect.top - 10, 0, -6)
        self.blocks = create_blocks()
        self.total_blocks = len(self.blocks)
        self.score = 0
        self.lives = 3

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self._init_game()
        
        # Iniciar la pelota automáticamente con velocidad horizontal
        self.ball.stuck = False
        self.ball.vx = 4.0
        self.ball.vy = -6.0

        return self._get_obs(), {}

    def _get_obs(self):
        # Normalización centrada entre -1.0 y 1.0 para mejor estabilidad del modelo
        paddle_x_norm = (self.paddle.rect.centerx / WIDTH) * 2 - 1
        ball_x_norm = (self.ball.x / WIDTH) * 2 - 1
        ball_y_norm = (self.ball.y / HEIGHT) * 2 - 1
        ball_vx_norm = self.ball.vx / 10.0
        ball_vy_norm = self.ball.vy / 10.0
        rel_x = (self.ball.x - self.paddle.rect.centerx) / WIDTH

        # Estado de los 60 bloques
        blocks_state = [1.0 if b.alive else 0.0 for b in self.blocks]

        state = [
            paddle_x_norm,
            ball_x_norm,
            ball_y_norm,
            ball_vx_norm,
            ball_vy_norm,
            rel_x
        ] + blocks_state

        return np.array(state, dtype=np.float32)

    def step(self, action):
        reward = 0.0
        terminated = False
        truncated = False

        # --- 1. ACCIONES ---
        if action == 1:   # Izquierda
            self.paddle.move(-1)
        elif action == 2: # Derecha
            self.paddle.move(1)

        if self.ball.stuck:
            self.ball.x = self.paddle.rect.centerx
            self.ball.y = self.paddle.rect.top - self.ball.r - 2

        self.ball.update()

        # --- 2. RECOMPENSAS Y COLISIONES ---
        
        # A) Recompensa por supervivencia
        reward += 0.01 

        # B) Recompensa de proximidad (guía para perseguir la pelota)
        dist_x = abs(self.paddle.rect.centerx - self.ball.x) / WIDTH
        reward += (1.0 - dist_x) * 0.01

        # C) Colisión con Paleta (Premia el atajar y dar ángulos)
        if not self.ball.stuck and self.ball.rect().colliderect(self.paddle.rect):
            hit_pos = (self.ball.x - self.paddle.rect.left) / self.paddle.rect.width
            self.ball.vx = (hit_pos - 0.5) * 10
            self.ball.vy = -abs(self.ball.vy)
            self.ball.y = self.paddle.rect.top - self.ball.r - 1
            
            # Bonus por pegarle con los bordes
            angle_bonus = abs(hit_pos - 0.5) * 2.0
            reward += 5.0 + angle_bonus

        # D) Colisión con Bloques
        if not self.ball.stuck:
            for block in self.blocks:
                if block.alive and self.ball.rect().colliderect(block.rect):
                    block.alive = False
                    self.score += 10
                    
                    # Multiplicador dinámico por avance del juego
                    destroyed_blocks = sum(1 for b in self.blocks if not b.alive)
                    progress_multiplier = 1.0 + (destroyed_blocks / self.total_blocks)
                    reward += 10.0 * progress_multiplier
                    
                    reflect_ball(self.ball, block.rect)
                    break

        # E) Caída de la Pelota
        if self.ball.y - self.ball.r > HEIGHT:
            self.lives -= 1
            reward -= 50.0  # Penalización fuerte por perder vida
            
            if self.lives <= 0:
                terminated = True
                reward -= 100.0
            else:
                self.ball.stuck = False
                self.ball.x = self.paddle.rect.centerx
                self.ball.y = self.paddle.rect.top - self.ball.r - 2
                self.ball.vx, self.ball.vy = 4.0, -6.0

        # F) Condición de Victoria
        if all(not b.alive for b in self.blocks):
            reward += 500.0
            terminated = True

        if self.render_mode == "human":
            self.render()

        return self._get_obs(), reward, terminated, truncated, {}

    def render(self):
        if self.screen is None:
            pygame.init()
            if self.render_mode == "human":
                self.screen = pygame.display.set_mode((WIDTH, HEIGHT))
                pygame.display.set_caption("Rompebloques - Entrenamiento IA")
            else:
                self.screen = pygame.Surface((WIDTH, HEIGHT))
        
        if self.clock is None:
            self.clock = pygame.time.Clock()

        self.screen.fill(BLACK)
        for y in range(0, HEIGHT, 40):
            pygame.draw.line(self.screen, (20, 20, 20), (0, y), (WIDTH, y))

        for b in self.blocks:
            b.draw(self.screen)

        self.paddle.draw(self.screen)
        self.ball.draw(self.screen)

        if self.render_mode == "human":
            pygame.event.pump()
            pygame.display.flip()
            self.clock.tick(60)

    def close(self):
        if self.screen is not None:
            pygame.quit()
            self.screen = None


# ==========================================
# 3. ENTRENAMIENTO EXPERTO DE LA IA
# ==========================================
if __name__ == "__main__":
    env = BreakoutEnv(render_mode=None)

    # Definir arquitectura profunda (red de 256x256)
    policy_kwargs = dict(net_arch=dict(pi=[256, 256], vf=[256, 256]))

    model = PPO(
        "MlpPolicy",
        env,
        learning_rate=0.0003,
        n_steps=4096,
        batch_size=128,
        gamma=0.99,
        policy_kwargs=policy_kwargs,
        verbose=1,
        tensorboard_log="./tb_logs/"
    )

    print("Iniciando entrenamiento experto de 800,000 pasos...")
    model.learn(total_timesteps=800000)
    
    model.save("bloques_ia_expert")
    print("¡Modelo Maestro guardado como 'bloques_ia_expert.zip'!")
    env.close()

Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
Iniciando entrenamiento experto de 800,000 pasos...
Logging to ./tb_logs/PPO_6
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 398      |
|    ep_rew_mean     | -193     |
| time/              |          |
|    fps             | 1288     |
|    iterations      | 1        |
|    time_elapsed    | 3        |
|    total_timesteps | 4096     |
---------------------------------
----------------------------------------
| rollout/                |            |
|    ep_len_mean          | 359        |
|    ep_rew_mean          | -204       |
| time/                   |            |
|    fps                  | 989        |
|    iterations           | 2          |
|    time_elapsed         | 8          |
|    total_timesteps      | 8192       |
| train/                  |            |
|    approx_kl            | 0.00861932 |
|    clip_fraction        | 0.0389  

In [15]:
model = PPO.load("bloques_ia_expert")

env = BreakoutEnv(render_mode="human")
obs, _ = env.reset()

for _ in range(10000):
    action, _states = model.predict(obs, deterministic=True)
    obs, reward, terminated, truncated, info = env.step(action)
    
    if terminated or truncated:
        print("Episodio terminado. Reiniciando...")
        obs, _ = env.reset()

env.close()

Episodio terminado. Reiniciando...
Episodio terminado. Reiniciando...
Episodio terminado. Reiniciando...
Episodio terminado. Reiniciando...
Episodio terminado. Reiniciando...
Episodio terminado. Reiniciando...
Episodio terminado. Reiniciando...
Episodio terminado. Reiniciando...
Episodio terminado. Reiniciando...
Episodio terminado. Reiniciando...
Episodio terminado. Reiniciando...
Episodio terminado. Reiniciando...
Episodio terminado. Reiniciando...
Episodio terminado. Reiniciando...
Episodio terminado. Reiniciando...
Episodio terminado. Reiniciando...
Episodio terminado. Reiniciando...
Episodio terminado. Reiniciando...


# MEJORA

In [30]:
import warnings
warnings.filterwarnings(
    "ignore",
    category=UserWarning,
    message=r".*pkg_resources is deprecated as an API.*"
)

import sys
import random
import pygame
import numpy as np
from dataclasses import dataclass
from typing import List

import gymnasium as gym
from gymnasium import spaces
from stable_baselines3 import PPO

# ==========================================
# 1. CONSTANTES Y CLASES DEL JUEGO
# ==========================================
WIDTH, HEIGHT = 800, 600

BLACK = (0, 0, 0)
WHITE = (240, 240, 240)
GRAY = (60, 60, 60)
BLUE = (80, 160, 255)
RED = (240, 90, 90)
GREEN = (80, 220, 120)
YELLOW = (250, 220, 100)
PURPLE = (170, 120, 250)
ORANGE = (255, 150, 70)

BLOCK_COLORS = [RED, ORANGE, YELLOW, GREEN, BLUE, PURPLE]

@dataclass
class Paddle:
    rect: pygame.Rect
    speed: int = 10

    def move(self, direction: int):
        self.rect.x += direction * self.speed
        self.rect.x = max(0, min(WIDTH - self.rect.width, self.rect.x))

    def draw(self, surf):
        pygame.draw.rect(surf, WHITE, self.rect, border_radius=6)

@dataclass
class Ball:
    x: float
    y: float
    vx: float
    vy: float
    r: int = 8
    stuck: bool = True

    def rect(self) -> pygame.Rect:
        return pygame.Rect(int(self.x - self.r), int(self.y - self.r), self.r * 2, self.r * 2)

    def update(self):
        if not self.stuck:
            self.x += self.vx
            self.y += self.vy

            if self.x - self.r <= 0 or self.x + self.r >= WIDTH:
                self.vx *= -1
                self.x = max(self.r, min(WIDTH - self.r, self.x))
            if self.y - self.r <= 0:
                self.vy *= -1
                self.y = self.r

    def draw(self, surf):
        pygame.draw.circle(surf, WHITE, (int(self.x), int(self.y)), self.r)

@dataclass
class Block:
    rect: pygame.Rect
    color: tuple
    alive: bool = True

    def draw(self, surf):
        if self.alive:
            pygame.draw.rect(surf, self.color, self.rect, border_radius=4)
            pygame.draw.rect(surf, GRAY, self.rect, width=2, border_radius=4)

def create_blocks(cols=10, rows=6, margin=6, top=70, height=24):
    blocks: List[Block] = []
    total_margin_x = margin * (cols + 1)
    block_w = (WIDTH - total_margin_x) // cols
    for r in range(rows):
        y = top + r * (height + margin)
        color = BLOCK_COLORS[r % len(BLOCK_COLORS)]
        for c in range(cols):
            x = margin + c * (block_w + margin)
            rect = pygame.Rect(x, y, block_w, height)
            blocks.append(Block(rect, color))
    return blocks

def reflect_ball(ball: Ball, rect: pygame.Rect) -> None:
    brect = ball.rect()
    overlap = brect.clip(rect)
    if overlap.width < overlap.height:
        ball.vx *= -1
        if brect.centerx < rect.centerx:
            ball.x = rect.left - ball.r
        else:
            ball.x = rect.right + ball.r
    elif overlap.height < overlap.width:
        ball.vy *= -1
        if brect.centery < rect.centery:
            ball.y = rect.top - ball.r
        else:
            ball.y = rect.bottom + ball.r
    else:
        ball.vx *= -1
        ball.vy *= -1

# ==========================================
# 2. ENTORNO GYMNASIUM ANTI-REPETICIÓN
# ==========================================
class BreakoutEnv(gym.Env):
    def __init__(self, render_mode=None):
        super().__init__()
        self.render_mode = render_mode
        self.screen = None
        self.clock = None

        self.action_space = spaces.Discrete(3)  # 0: Quieto, 1: Izq, 2: Der

        num_blocks = 60
        # 6 variables + 60 bloques
        self.observation_space = spaces.Box(
            low=-1.0, high=1.0, shape=(6 + num_blocks,), dtype=np.float32
        )

        self._init_game()

    def _init_game(self):
        paddle_w, paddle_h = 110, 16
        self.paddle = Paddle(pygame.Rect((WIDTH - paddle_w) // 2, HEIGHT - 60, paddle_w, paddle_h))
        self.ball = Ball(self.paddle.rect.centerx, self.paddle.rect.top - 10, 0, -6)
        self.blocks = create_blocks()
        self.total_blocks = len(self.blocks)
        self.score = 0
        self.lives = 3

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self._init_game()
        
        # INICIO ALEATORIO: Evita que la IA memorice un único patrón fijo
        self.ball.stuck = False
        self.ball.vx = random.choice([-6.0, -4.0, -2.0, 2.0, 4.0, 6.0])
        self.ball.vy = -6.0

        return self._get_obs(), {}

    def _get_obs(self):
        paddle_x_norm = (self.paddle.rect.centerx / WIDTH) * 2 - 1
        ball_x_norm = (self.ball.x / WIDTH) * 2 - 1
        ball_y_norm = (self.ball.y / HEIGHT) * 2 - 1
        ball_vx_norm = self.ball.vx / 10.0
        ball_vy_norm = self.ball.vy / 10.0
        rel_x = (self.ball.x - self.paddle.rect.centerx) / WIDTH

        blocks_state = [1.0 if b.alive else 0.0 for b in self.blocks]

        state = [
            paddle_x_norm,
            ball_x_norm,
            ball_y_norm,
            ball_vx_norm,
            ball_vy_norm,
            rel_x
        ] + blocks_state

        return np.array(state, dtype=np.float32)

    def step(self, action):
        reward = -0.02  # Penalización por tiempo: incentiva a limpiar rápido
        terminated = False
        truncated = False

        if action == 1:
            self.paddle.move(-1)
        elif action == 2:
            self.paddle.move(1)

        if self.ball.stuck:
            self.ball.x = self.paddle.rect.centerx
            self.ball.y = self.paddle.rect.top - self.ball.r - 2

        self.ball.update()

        # Guía básica para estar bajo la pelota
        dist_x = abs(self.paddle.rect.centerx - self.ball.x) / WIDTH
        reward += (1.0 - dist_x) * 0.01

        # Recompensa por atajar
        if not self.ball.stuck and self.ball.rect().colliderect(self.paddle.rect):
            hit_pos = (self.ball.x - self.paddle.rect.left) / self.paddle.rect.width
            self.ball.vx = (hit_pos - 0.5) * 10
            self.ball.vy = -abs(self.ball.vy)
            self.ball.y = self.paddle.rect.top - self.ball.r - 1
            
            # Bonus si busca un tiro en ángulo
            angle_bonus = abs(hit_pos - 0.5) * 3.0
            reward += 3.0 + angle_bonus

        # Recompensa por romper bloques (Aumenta progresivamente)
        if not self.ball.stuck:
            for block in self.blocks:
                if block.alive and self.ball.rect().colliderect(block.rect):
                    block.alive = False
                    self.score += 10
                    
                    destroyed_blocks = sum(1 for b in self.blocks if not b.alive)
                    progress = (destroyed_blocks / self.total_blocks)
                    reward += 15.0 * (1.0 + progress)
                    
                    reflect_ball(self.ball, block.rect)
                    break

        # Caída de la Pelota: PENALIZACIÓN SEVERA PARA EVITAR PERDER VIDAS
        if self.ball.y - self.ball.r > HEIGHT:
            self.lives -= 1
            reward -= 150.0  # El castigo por perder vida arruina el episodio
            
            if self.lives <= 0:
                terminated = True
                reward -= 100.0
            else:
                self.ball.stuck = False
                self.ball.x = self.paddle.rect.centerx
                self.ball.y = self.paddle.rect.top - self.ball.r - 2
                self.ball.vx = random.choice([-5.0, -3.0, 3.0, 5.0])
                self.ball.vy = -6.0

        # Condición de Victoria Total
        if all(not b.alive for b in self.blocks):
            # Gran recompensa + bonus si mantuvo las 3 vidas intactas
            no_loss_bonus = 300.0 if self.lives == 3 else 0.0
            reward += 1000.0 + no_loss_bonus
            terminated = True

        if self.render_mode == "human":
            self.render()

        return self._get_obs(), reward, terminated, truncated, {}

    def render(self):
        if self.screen is None:
            pygame.init()
            if self.render_mode == "human":
                self.screen = pygame.display.set_mode((WIDTH, HEIGHT))
                pygame.display.set_caption("Rompebloques - Entrenamiento Dinámico")
            else:
                self.screen = pygame.Surface((WIDTH, HEIGHT))
        
        if self.clock is None:
            self.clock = pygame.time.Clock()

        self.screen.fill(BLACK)
        for y in range(0, HEIGHT, 40):
            pygame.draw.line(self.screen, (20, 20, 20), (0, y), (WIDTH, y))

        for b in self.blocks:
            b.draw(self.screen)

        self.paddle.draw(self.screen)
        self.ball.draw(self.screen)

        if self.render_mode == "human":
            pygame.event.pump()
            pygame.display.flip()
            self.clock.tick(60)

    def close(self):
        if self.screen is not None:
            pygame.quit()
            self.screen = None


# ==========================================
# 3. ENTRENAMIENTO AVANZADO
# ==========================================
if __name__ == "__main__":
    env = BreakoutEnv(render_mode=None)

    policy_kwargs = dict(net_arch=dict(pi=[256, 256], vf=[256, 256]))

    model = PPO(
        "MlpPolicy",
        env,
        learning_rate=0.0003,
        n_steps=4096,
        batch_size=128,
        gamma=0.99,
        ent_coef=0.01,  # FORZA LA EXPLORACIÓN para evitar repetir el mismo movimiento
        policy_kwargs=policy_kwargs,
        verbose=1,
        tensorboard_log="./tb_logs/"
    )

    print("Iniciando entrenamiento avanzado (1,000,000 de pasos)...")
    model.learn(total_timesteps=10000000)
    
    model.save("bloques_ia_dinamica")
    print("¡Modelo Guardado como 'bloques_ia_dinamica.zip'!")
    env.close()

Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
Iniciando entrenamiento avanzado (1,000,000 de pasos)...
Logging to ./tb_logs/PPO_9
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 353      |
|    ep_rew_mean     | -499     |
| time/              |          |
|    fps             | 1205     |
|    iterations      | 1        |
|    time_elapsed    | 3        |
|    total_timesteps | 4096     |
---------------------------------
----------------------------------------
| rollout/                |            |
|    ep_len_mean          | 361        |
|    ep_rew_mean          | -498       |
| time/                   |            |
|    fps                  | 936        |
|    iterations           | 2          |
|    time_elapsed         | 8          |
|    total_timesteps      | 8192       |
| train/                  |            |
|    approx_kl            | 0.00115208 |
|    clip_fraction        | 0.0

In [47]:
model = PPO.load("bloques_ia_dinamica")

env = BreakoutEnv(render_mode="human")
obs, _ = env.reset()

for _ in range(10000):
    action, _states = model.predict(obs, deterministic=True)
    obs, reward, terminated, truncated, info = env.step(action)
    
    if terminated or truncated:
        print("Fin de partida. Iniciando nueva partida...")
        obs, _ = env.reset()

env.close()

Fin de partida. Iniciando nueva partida...
Fin de partida. Iniciando nueva partida...
Fin de partida. Iniciando nueva partida...
Fin de partida. Iniciando nueva partida...


In [53]:
import pygame
from stable_baselines3 import PPO

# 1. Cargar el modelo entrenado
model = PPO.load("bloques_ia_dinamica")

# 2. Crear el entorno visual
env = BreakoutEnv(render_mode="human")

num_partidas = 5  # Número de partidas completas a visualizar

for partida in range(num_partidas):
    print(f"--- Iniciando Partida {partida + 1} de {num_partidas} ---")
    
    obs, _ = env.reset()
    env.render()  # Forzar la inicialización de la pantalla y el subsistema de video
    
    jugando = True
    
    while jugando:
        # Procesar la cola de eventos de Pygame
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                jugando = False
                num_partidas = 0
                break

        if not jugando:
            break

        # Predicción y paso en el entorno
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, _ = env.step(action)
        
        # Verificar fin de episodio
        if terminated or truncated:
            if all(not b.alive for b in env.blocks):
                print(f"¡Partida {partida + 1} GANADA! Puntuación: {env.score}")
            else:
                print(f"Partida {partida + 1} finalizada. Puntuación: {env.score}")
            
            jugando = False

env.close()
print("Demostración finalizada con éxito.")

--- Iniciando Partida 1 de 5 ---
Partida 1 finalizada. Puntuación: 200
--- Iniciando Partida 2 de 5 ---
Partida 2 finalizada. Puntuación: 210
--- Iniciando Partida 3 de 5 ---
Partida 3 finalizada. Puntuación: 240
--- Iniciando Partida 4 de 5 ---
Partida 4 finalizada. Puntuación: 400
--- Iniciando Partida 5 de 5 ---
Partida 5 finalizada. Puntuación: 260
Demostración finalizada con éxito.
